In [63]:
# Imports
import os
import requests
import pandas as pd
from dotenv import load_dotenv
import json
import re

In [64]:
load_dotenv()

api_key = os.getenv("DOG_API_KEY")
url="https://api.thedogapi.com/v1/breeds"
headers = {"x-api-key": api_key}

response = requests.get(url, headers=headers)
data = response.json()



In [65]:
clean_data = []
for breed in data:
    clean_data.append({
        "id": breed.get("id"),
        "name": breed.get("name"),
        "life_span": breed.get("life_span"),
        "temperament": breed.get("temperament"),
        "origin": breed.get("origin"),
        "country_code": breed.get("country_code"),
        "breed_group": breed.get("breed_group"),
        "description": breed.get("description"),
        "history": breed.get("history"),
        "weight_metric": breed.get("weight", {}).get("metric"),
        "height_metric": breed.get("height", {}).get("metric"),
        "reference_image_id": breed.get("reference_image_id")
    })

# Save raw data
os.makedirs("/app/data/", exist_ok=True)

with open("data/dog_breeds_raw.json", "w", encoding="utf-8") as file:
    json.dump(data, file, ensure_ascii=False, indent=4)

print(f"Saved {len(data)} breeds.")

Saved 631 breeds.


In [66]:
# Load dataset
df = pd.read_json("data/dog_breeds_raw.json")

print(df.shape)
print(df.info())

(631, 17)
<class 'pandas.DataFrame'>
RangeIndex: 631 entries, 0 to 630
Data columns (total 17 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id                  631 non-null    int64  
 1   name                631 non-null    str    
 2   species_id          631 non-null    int64  
 3   life_span           591 non-null    str    
 4   temperament         631 non-null    str    
 5   origin              631 non-null    str    
 6   country_codes       631 non-null    str    
 7   country_code        631 non-null    str    
 8   description         631 non-null    str    
 9   bred_for            0 non-null      float64
 10  perfect_for         0 non-null      float64
 11  breed_group         631 non-null    str    
 12  history             631 non-null    str    
 13  reference_image_id  631 non-null    str    
 14  weight              631 non-null    object 
 15  height              631 non-null    object 
 16  image    

In [67]:
# Check for duplicates
print("Duplicated IDs:", df["id"].duplicated().sum())
print("Duplicated names:", df["name"].duplicated().sum())

duplicates = df[df["name"].duplicated(keep=False)]

print(duplicates[["id", "name", "breed_group", "origin"]])

Duplicated IDs: 0
Duplicated names: 1
      id                    name breed_group              origin
146  269  Caucasian Shepherd Dog     Working  Caucasus Mountains
147   70  Caucasian Shepherd Dog     Working  Caucasus Mountains


In [68]:
# Drop duplicates
df = df.drop_duplicates(subset="name", keep="first")

In [69]:
# Check for null values
missing = pd.DataFrame({
    "missing_count": df.isnull().sum(),
    "missing_pct": df.isnull().mean() * 100})

missing = missing.sort_values("missing_pct", ascending=False)
print(missing)

                    missing_count  missing_pct
perfect_for                   630   100.000000
bred_for                      630   100.000000
life_span                      40     6.349206
id                              0     0.000000
name                            0     0.000000
origin                          0     0.000000
country_codes                   0     0.000000
species_id                      0     0.000000
temperament                     0     0.000000
description                     0     0.000000
country_code                    0     0.000000
breed_group                     0     0.000000
history                         0     0.000000
reference_image_id              0     0.000000
weight                          0     0.000000
height                          0     0.000000
image                           0     0.000000


In [70]:
# Drop null and duplicated columns
df.drop(columns=["perfect_for", "bred_for", "country_codes"], inplace=True)

In [71]:

print(df[["id", "name", "breed_group", "origin"]].nunique())

id             630
name           630
breed_group     27
origin         315
dtype: int64


In [72]:
df.shape

(630, 14)

In [73]:
# Function to parse ranges of weight and height
def parse_range(value, unit="metric"):
    if not isinstance(value, dict):
        return pd.Series([None, None, None])
    metric = value.get(unit)
    if not metric:
        return pd.Series([None, None, None])
    numbers = [float(x) for x in re.findall(r"\d+(?:\.\d+)?", metric)]
    if not numbers:
        return pd.Series([None, None, None])
    min_value = min(numbers)
    max_value = max(numbers)
    midpoint = (min_value + max_value) / 2
    return pd.Series([min_value, max_value, midpoint])

In [74]:
# Parse weight - get min, max and calculate average from metric system
df[["weight_min_kg", "weight_max_kg", "weight_avg_kg"]] = (df["weight"].apply(parse_range))

# Parse height - get min, max and calculate average from metric system
df[["height_min_cm", "height_max_cm", "height_mid_cm"]] = (df["height"].apply(parse_range))

print(df[["weight_min_kg", "weight_max_kg", "height_min_cm", "height_max_cm"]].describe())

       weight_min_kg  weight_max_kg  height_min_cm  height_max_cm
count     628.000000     628.000000     628.000000     628.000000
mean       17.939331      29.369586      44.843312      55.722930
std        10.959591      18.450794      13.596919      15.360645
min         1.000000       2.700000      13.000000      15.000000
25%         9.000000      16.000000      36.000000      46.000000
50%        17.000000      27.000000      46.000000      58.000000
75%        23.000000      36.000000      56.000000      67.000000
max        59.000000     104.000000      76.000000      91.000000


In [75]:
df.shape

(630, 20)

In [76]:
# Parse life span
df[["life_span_min_years", "life_span_max_years"]] = (df["life_span"].str.split("-", expand=True))
df["life_span_min_years"] = pd.to_numeric(df["life_span_min_years"], errors="coerce")
df["life_span_max_years"] = pd.to_numeric(df["life_span_max_years"], errors="coerce")
df["life_span_mid_years"] = (df["life_span_min_years"] + df["life_span_max_years"]) / 2

In [77]:
# Inspecting temperament
temperaments = (df["temperament"]
    .dropna()
    .str.split(",")
    .explode()
    .str.strip()
    .str.lower())
print("Unique temperament terms:", temperaments.nunique())
print(temperaments.value_counts())

Unique temperament terms: 49
temperament
intelligent                                             538
loyal                                                   454
alert                                                   376
energetic                                               304
courageous                                              212
independent                                             210
affectionate                                            165
friendly                                                163
playful                                                 142
protective                                              142
confident                                               138
gentle                                                  129
calm                                                    125
work-focused                                            103
devoted                                                  78
adaptable                                                50

In [78]:
odd_temp = df[df["temperament"].str.contains(
    "variable depending on ancestry",
    case=False,
    na=False
)]
print(odd_temp["name"])

term_to_remove = "variable depending on ancestry and individual traits"

df["temperament"] = (df["temperament"].apply(lambda x: ", ".join(term.strip() for term in x.split(",") if term.strip().lower() != term_to_remove)
            if pd.notna(x) else x))

df["temperament"] = [x.strip().lower() for x in df["temperament"]]
temperaments = (df["temperament"]
    .dropna()
    .str.split(",")
    .explode()
    .str.strip()
    .str.lower())
temperaments = temperaments[temperaments != ""]
print(df[
    df["temperament"]
    .fillna("")
    .str.contains(
        "variable depending on ancestry",
        case=False
    )
])
print("Unique temperament terms:", temperaments.nunique())
print(sorted(temperaments.unique()))

389    Mongrel
Name: name, dtype: str
Empty DataFrame
Columns: [id, name, species_id, life_span, temperament, origin, country_code, description, breed_group, history, reference_image_id, weight, height, image, weight_min_kg, weight_max_kg, weight_avg_kg, height_min_cm, height_max_cm, height_mid_cm, life_span_min_years, life_span_max_years, life_span_mid_years]
Index: []

[0 rows x 23 columns]
Unique temperament terms: 48
['active', 'adaptable', 'affectionate', 'agile', 'alert', 'aloof', 'amiable', 'athletic', 'calm', 'cautious', 'charming', 'cheerful', 'clever', 'confident', 'courageous', 'curious', 'determined', 'devoted', 'dignified', 'docile', 'eager to please', 'easygoing', 'energetic', 'even-tempered', 'fearless', 'friendly', 'gentle', 'good-natured', 'happy', 'hardy', 'independent', 'intelligent', 'lively', 'loyal', 'merry', 'mischievous', 'optimistic', 'outgoing', 'patient', 'playful', 'protective', 'reserved', 'sensitive', 'smart', 'spirited', 'sweet-tempered', 'tenacious', 'wo

In [79]:
df.head()

,id,name,species_id,life_span,temperament,origin,country_code,description,breed_group,history,...,image,weight_min_kg,weight_max_kg,weight_avg_kg,height_min_cm,height_max_cm,height_mid_cm,life_span_min_years,life_span_max_years,life_span_mid_years
0,1,Affenpinscher,2,12-15,"confident, alert, playful, loyal, courageous",Germany,DE,"Small, sturdy toy breed with a distinctive mon...",Toy,"Originating in 17th-century Germany, bred down...",...,"{'id': '0LJiOVlxp', 'url': 'https://cdn2.thedo...",3.2,4.5,3.85,23.0,29.0,26.0,12.0,15.0,13.5
1,2,Afghan Hound,2,12-15,"independent, dignified, aloof, loyal, confident",Afghanistan,AF,"Ancient, elegant sighthound with dramatic, flo...",Hound,Ancient sighthound originating in the mountain...,...,"{'id': 'tChrH8dDJ', 'url': 'https://cdn2.thedo...",20.0,30.0,25.00,63.0,74.0,68.5,12.0,15.0,13.5
2,347,Africanis,2,10-15,"intelligent, loyal, alert, adaptable, independent",Southern Africa,ZA,"Medium-sized, athletic landrace dog indigenous...",Primitive,Ancient landrace breed indigenous to Southern ...,...,"{'id': 'OAr3CJcw9i', 'url': 'https://cdn4.thed...",20.0,34.0,27.00,48.0,60.0,54.0,10.0,15.0,12.5
3,348,Aidi,2,10-12,"alert, protective, independent, loyal, courage...","Atlas Mountains, Morocco",MA,"Medium to large, powerful and muscular Morocca...",Working,Ancient Berber breed from the Atlas Mountains ...,...,"{'id': 'dK6eTCqT2i', 'url': 'https://storage.g...",23.0,25.0,24.00,46.0,62.0,54.0,10.0,12.0,11.0
4,4,Airedale Terrier,2,11-14,"confident, intelligent, courageous, alert, ene...","Yorkshire, England",GB,"The largest of all terrier breeds, the Airedal...",Terrier,"Developed in the Aire Valley of Yorkshire, Eng...",...,"{'id': 'QWRBrrIvB', 'url': 'https://cdn2.thedo...",18.0,32.0,25.00,56.0,61.0,58.5,11.0,14.0,12.5


In [80]:
df_clean = df[['name', 'species_id', 'temperament', 'origin','country_code', 'description', 'breed_group', 'history',
       'image', 'weight_min_kg','weight_max_kg', 'weight_avg_kg', 'height_min_cm', 'height_max_cm',
       'height_mid_cm', 'life_span_min_years', 'life_span_max_years', 'life_span_mid_years']]

In [81]:
df_clean.to_csv("data\\dog_breeds_dog_api_clean.csv", index=False)